# Video Frame Prediction using Convolutional LSTM

This cleaned portfolio notebook reproduces and explains the attached experiment using the modular project code. The task is **single-step next-frame prediction** from six 32 × 32 grayscale frames. A recursive rollout is also demonstrated for educational multi-step forecasting.

> **Responsible use:** The model is an educational synthetic-data demonstration. Do not use it for surveillance, medical, autonomous-driving, legal, safety-critical, or production monitoring decisions.

## 1. Setup

The deployment uses Keras 3 with the JAX backend. Set the backend before importing Keras.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODEL_PATH, METADATA_PATH
from src.model_evaluation import (
    build_comparison_table,
    frame_average_baseline,
    persistence_baseline,
)
from src.prediction_pipeline import batch_predict, load_prediction_model, recursive_predict
from src.sequence_generation import generate_moving_sequences
from src.visualization import plot_input_sequence, plot_prediction_comparison

SEED = 42
print(PROJECT_ROOT)

## 2. Reproduce the synthetic moving-object dataset

The original notebook creates 2,500 independent sequences. Each input contains six ordered frames, and the target is the seventh frame.

In [ ]:
X_all, y_all = generate_moving_sequences(
    n_samples=2500,
    seq_len=6,
    image_size=32,
    obj_size=5,
    seed=SEED,
)
print("Inputs:", X_all.shape)
print("Targets:", y_all.shape)

## 3. Train, validation, and test split

Frames are never shuffled within a sequence. The split occurs across complete independent sequences to avoid temporal leakage between windows.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 4. Inspect one ordered sequence

In [ ]:
figure = plot_input_sequence(X_train[0])
plt.show()

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(y_train[0, ..., 0], cmap="gray", vmin=0, vmax=1)
plt.title("Target next frame")
plt.axis("off")
plt.show()

## 5. Load the supplied trained model

The model contains two ConvLSTM layers followed by convolutional reconstruction layers. It has 117,025 parameters and predicts one frame.

In [ ]:
model = load_prediction_model(MODEL_PATH)
model.summary()

## 6. Reproduce test predictions and baselines

The persistence baseline copies the last input frame. The frame-average baseline averages all six observed frames.

In [ ]:
test_predictions = batch_predict(model, X_test, batch_size=32)
comparison = build_comparison_table(
    y_test,
    {
        "Persistence (last frame)": persistence_baseline(X_test),
        "Frame average": frame_average_baseline(X_test),
        "ConvLSTM": test_predictions,
    },
)
comparison

### Interpretation

ConvLSTM strongly improves MAE, RMSE, PSNR, foreground IoU, and thresholded pixel accuracy. SSIM favors persistence because most pixels are static black background; therefore all metrics and visual outputs should be interpreted together.

## 7. Visualize actual versus predicted frame

In [ ]:
sample_index = 0
figure = plot_prediction_comparison(
    X_test[sample_index],
    test_predictions[sample_index],
    y_test[sample_index],
)
plt.show()

## 8. Best and worst single-step examples

In [ ]:
sample_mae = np.mean(np.abs(y_test - test_predictions), axis=(1, 2, 3))
best = np.argsort(sample_mae)[:3]
worst = np.argsort(sample_mae)[-3:]
print("Best indices:", best)
print("Worst indices:", worst)

for label, indices in [("Best", best), ("Worst", worst)]:
    print(label)
    for index in indices:
        figure = plot_prediction_comparison(X_test[index], test_predictions[index], y_test[index])
        plt.show()

## 9. Recursive multi-step forecasting

The trained model predicts one frame. To generate a longer sequence, each predicted frame is appended to the rolling six-frame window. This is convenient but causes error accumulation.

In [ ]:
long_input, actual_future = generate_moving_sequences(
    n_samples=1,
    seq_len=6,
    future_frames=6,
    seed=2026,
)
recursive_future = recursive_predict(model, long_input[0], future_steps=6)

figure, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(6):
    axes[0, i].imshow(actual_future[0, i, ..., 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"Actual +{i+1}")
    axes[0, i].axis("off")
    axes[1, i].imshow(recursive_future[i, ..., 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Predicted +{i+1}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 10. Metadata and deployment artifacts

In [ ]:
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
metadata

## 11. Key limitations

- Synthetic 5 × 5 moving squares are much simpler than real video.
- Real uploads are out-of-distribution and should be treated only as an inference-pipeline demonstration.
- Pixel accuracy is inflated by the large background area.
- Recursive rollout compounds one-step prediction errors.
- ConvLSTM is an educational baseline; modern video prediction may use transformer, latent, diffusion, or probabilistic architectures.

## 12. Next steps

1. Train on a clearly licensed Moving MNIST or real-world dataset.
2. Use direct sequence-to-sequence training for multiple future frames.
3. Add foreground-weighted or structure-aware loss functions.
4. Compare against CNN-LSTM, PredRNN, SimVP, transformer, and diffusion baselines.
5. Evaluate object-centroid displacement and temporal consistency.